# ValoMind: five full VCT playoff broadcasts on one T4
Run all cells. The queue collects proposed OCR/HUD context and unlabeled review samples
from five official archived Champions Seoul 2024 broadcasts (4h44m–6h37m; ~27h20m total).
It does **not** train a model, infer thoughts, or claim verified player/agent identities.

Colab: select Runtime → Change runtime type → T4 GPU. Drive stores metadata, SQLite
checkpoints and images; only the current full video uses temporary Colab disk.
Run the queue cell again after its budget expires. After a disconnect, rerun setup/config
with the same RUN_NAME and parameters. Never run two notebooks against the same queue root.
Free GPU availability, runtime duration and YouTube access are not guaranteed.

New broadcasts have no reviewed player catalog/HUD calibration. Empty roster exports are
expected until those are supplied. Full-frame OCR and review images still provide data.


In [ ]:
import sys, subprocess, json, hashlib, math, shutil, csv
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "-q",
        "yt-dlp[default]>=2026.1",
        "opencv-python-headless>=4.10,<5",
        "easyocr>=1.7,<2",
        "ultralytics>=8.3,<9",
    ],
    check=True,
)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("This runtime needs ffmpeg and ffprobe.")
MODULES = {
    "__init__.py": r'''"""Portable notebook processing; heavy GPU dependencies are loaded only when used."""
''',
    "acquisition.py": r'''"""Supported JS runtime setup and bounded, explicit YouTube authentication options."""

import hashlib
import io
import json
import os
import platform
import re
import shutil
import subprocess
import urllib.request
from pathlib import Path
from zipfile import ZipFile


def detect_runtime():
    for name, minimum in [("deno", (2, 3, 0)), ("node", (22, 0, 0))]:
        executable = shutil.which(name)
        if not executable:
            continue
        try:
            version = subprocess.check_output([executable, "--version"], text=True, timeout=10)
            match = re.search(r"(\d+)\.(\d+)\.(\d+)", version)
            if match and tuple(map(int, match.groups())) >= minimum:
                return {name: {"path": executable}}
        except (OSError, subprocess.SubprocessError):
            continue
    return {}


def ensure_runtime():
    """Install the official Deno Linux binary in the notebook, checking its asset digest."""
    found = detect_runtime()
    if found:
        return found
    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("Install Deno >=2.3 or Node >=22 and rerun setup")
    request = urllib.request.Request(
        "https://api.github.com/repos/denoland/deno/releases/latest",
        headers={"User-Agent": "ValoMind-notebook"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        release = json.load(response)
    asset = next(
        (item for item in release["assets"] if item["name"] == "deno-x86_64-unknown-linux-gnu.zip"),
        None,
    )
    if not asset or not str(asset.get("digest", "")).startswith("sha256:"):
        raise RuntimeError("Official Deno asset has no verification digest; install Deno manually")
    url = asset["browser_download_url"]
    if not url.startswith("https://github.com/denoland/deno/releases/download/"):
        raise RuntimeError("Unexpected Deno release asset location")
    with urllib.request.urlopen(url, timeout=120) as response:
        data = response.read()
    if hashlib.sha256(data).hexdigest() != asset["digest"].split(":", 1)[1]:
        raise RuntimeError("Deno asset checksum mismatch")
    target = Path.home() / ".local/bin/deno"
    target.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(io.BytesIO(data)) as archive:
        binary = archive.read("deno")
    temporary = target.with_suffix(".tmp")
    temporary.write_bytes(binary)
    temporary.chmod(0o755)
    temporary.replace(target)
    os.environ["PATH"] = str(target.parent) + os.pathsep + os.environ.get("PATH", "")
    found = detect_runtime()
    if not found:
        raise RuntimeError("Deno installation did not pass the runtime version check")
    return found


def youtube_options(cookie_file=None, js_runtime=None, export_root=None):
    options = {"js_runtimes": js_runtime or detect_runtime()}
    if cookie_file:
        path = Path(cookie_file).expanduser().resolve()
        if not path.is_file():
            raise ValueError("COOKIE_FILE does not exist in this cloud runtime")
        if export_root and path.is_relative_to(Path(export_root).resolve()):
            raise ValueError(
                "Keep COOKIE_FILE outside WORK_ROOT so it cannot enter result archives"
            )
        options["cookiefile"] = str(path)
    return options


def download_error_message(error):
    message = str(error).casefold()
    if "not a bot" in message or "sign in to confirm" in message:
        return (
            "YouTube requires sign-in/verification from this cloud runtime. The JS runtime "
            "alone cannot guarantee access. Stop retrying; optionally configure COOKIE_FILE "
            "from your own authorized browser session outside WORK_ROOT, or reuse the "
            "previously downloaded video in persistent cloud storage. Never paste cookies "
            "into chat or commit them. No automatic proxy/account rotation is attempted."
        )
    return "YouTube acquisition failed. Check video availability, runtime setup and the downloader logs."


def cached_video(folder, key):
    """Reuse only a decodable completed video with matching stored source metadata."""
    info = folder / "video.info.json"
    if not info.is_file():
        return None
    try:
        metadata = json.loads(info.read_text(encoding="utf-8"))
        if metadata.get("id") != key or not shutil.which("ffprobe"):
            return None
        for suffix in [".mp4", ".webm", ".mkv"]:
            video = folder / ("video" + suffix)
            if not video.is_file() or video.stat().st_size == 0:
                continue
            probe = subprocess.check_output(
                [
                    "ffprobe",
                    "-v",
                    "error",
                    "-show_entries",
                    "format=duration",
                    "-of",
                    "json",
                    str(video),
                ],
                text=True,
                timeout=30,
            )
            duration = float(json.loads(probe)["format"]["duration"])
            expected = float(metadata.get("duration") or 0)
            if duration > 0 and (
                not expected or abs(duration - expected) <= max(2, expected * 0.01)
            ):
                return video
    except (OSError, ValueError, KeyError, subprocess.SubprocessError):
        return None
    return None


def restore_source_archive(archive_path, root, key):
    """Restore only the selected prior source video and metadata, never arbitrary ZIP paths."""
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", key):
        raise ValueError("Invalid source video ID")
    folder = Path(root) / "sources" / key
    existing = cached_video(folder, key)
    if existing:
        return existing
    prefix = f"sources/{key}/"
    with ZipFile(archive_path) as archive:
        names = set(archive.namelist())
        info_name = prefix + "video.info.json"
        video_name = next(
            (
                prefix + "video" + suffix
                for suffix in [".mp4", ".webm", ".mkv"]
                if prefix + "video" + suffix in names
            ),
            None,
        )
        if not video_name or info_name not in names:
            raise ValueError(
                "This archive has no full source video. Use the FIRST valomind_results.zip, not the later metadata-only exports."
            )
        if archive.getinfo(info_name).file_size > 5_000_000:
            raise ValueError("Source metadata is unexpectedly large")
        metadata_bytes = archive.read(info_name)
        metadata = json.loads(metadata_bytes)
        if metadata.get("id") != key:
            raise ValueError("Archive metadata does not match the requested YouTube video")
        expected = set()
        for name in names:
            if name.startswith("jobs/") and name.endswith("/manifest.json"):
                if archive.getinfo(name).file_size > 1_000_000:
                    raise ValueError("Unexpectedly large job manifest")
                manifest = json.loads(archive.read(name))
                digest = manifest.get("source_sha256", "")
                if manifest.get("video_id") == key and re.fullmatch(r"[a-f0-9]{64}", digest):
                    expected.add(digest)
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / Path(video_name).name
        temporary = folder / (destination.name + ".restore.part")
        digest = hashlib.sha256()
        try:
            with archive.open(video_name) as source, temporary.open("wb") as output:
                for chunk in iter(lambda: source.read(4 * 1024 * 1024), b""):
                    digest.update(chunk)
                    output.write(chunk)
            if temporary.stat().st_size == 0 or (expected and digest.hexdigest() not in expected):
                raise ValueError("Restored source checksum does not match the archived job")
            temporary.replace(destination)
            info = folder / "video.info.json"
            info_temp = folder / "video.info.json.restore.part"
            info_temp.write_bytes(metadata_bytes)
            info_temp.replace(info)
        finally:
            if temporary.exists():
                temporary.unlink()
    restored = cached_video(folder, key)
    if not restored:
        raise ValueError(
            "Restored video failed ffprobe validation; check ffmpeg/ffprobe setup and archive completeness"
        )
    print(
        "Restored prior source into the cloud cache; YouTube access is not needed for this video."
    )
    return restored
''',
    "batch.py": r'''"""Persistent, bounded, single-worker broadcast queue. No accounts or GPU parallelism."""

import hashlib
import json
import math
import time
from pathlib import Path

from .pipeline import video_id

OFFICIAL_CHANNEL = "UCA1d3HFGFUmkKr2JIUA5Vlw"


def validate_sources(sources):
    if not 1 <= len(sources) <= 5:
        raise ValueError("Select one to five archived broadcasts")
    seen = set()
    for source in sources:
        key = video_id(source["url"])
        if key != source["video_id"] or key in seen:
            raise ValueError("Duplicate or mismatched source ID")
        seen.add(key)
        if source["channel_id"] != OFFICIAL_CHANNEL:
            raise ValueError("Expected the official VALORANT Champions Tour channel")
        if source.get("is_live_now") or not source.get("is_live_content"):
            raise ValueError("Use completed livestream archives, not active streams or highlights")
        if not 4 * 3600 <= source["duration_seconds"] <= 12 * 3600:
            raise ValueError("Expected a long broadcast between four and twelve hours")
        if source["dataset_role"] not in {"train_proposed", "validation_reserved", "test_reserved"}:
            raise ValueError("Unknown dataset role")
    return sources


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def run_queue(
    sources,
    root,
    worker,
    configuration,
    budget_seconds=5400,
    retry_errors=False,
    clock=time.monotonic,
):
    """Worker returns a partial/completed output CSV. Checkpoints survive cell interruption.

    Budget is checked between worker chunks, not during downloads/inference. Only
    one notebook may own a queue root. Error retries require explicit configuration.
    """
    validate_sources(sources)
    if not math.isfinite(budget_seconds) or budget_seconds <= 0:
        raise ValueError("Positive finite runtime budget required")
    root = Path(root).resolve()
    path = root / "queue_status.json"
    signature = hashlib.sha256(
        json.dumps({"sources": sources, "configuration": configuration}, sort_keys=True).encode()
    ).hexdigest()
    if path.exists():
        state = json.loads(path.read_text(encoding="utf-8"))
        if state["signature"] != signature:
            raise ValueError(
                "Queue configuration changed. Use a new RUN_NAME/root to preserve old jobs"
            )
    else:
        state = {
            "schema_version": "broadcast_queue_v1",
            "signature": signature,
            "configuration": configuration,
            "jobs": {row["video_id"]: {"status": "pending", "chunks": 0} for row in sources},
        }
    if retry_errors:
        for record in state["jobs"].values():
            if record["status"] in {"failed", "blocked"}:
                record["status"] = "paused"
    for record in state["jobs"].values():
        if record["status"] == "complete":
            candidate = (root / record.get("output_csv", "")).resolve()
            if not candidate.is_relative_to(root) or not candidate.is_file():
                record["status"] = "paused"
    atomic_json(path, state)
    deadline = clock() + budget_seconds
    while clock() < deadline:
        if any(row["status"] in {"failed", "blocked"} for row in state["jobs"].values()):
            print(
                "Queue stopped on an error. Fix acquisition/configuration before enabling RETRY_ERRORS."
            )
            break
        source = next(
            (row for row in sources if state["jobs"][row["video_id"]]["status"] != "complete"), None
        )
        if source is None:
            break
        record = state["jobs"][source["video_id"]]
        record["status"] = "running"
        atomic_json(path, state)
        try:
            print("Processing", source["video_id"], source["title"], flush=True)
            result = worker(source)
            if not isinstance(result.get("complete"), bool):
                raise TypeError("Worker must explicitly report boolean completion")
            output = Path(result["output_csv"]).resolve()
            if not output.is_relative_to(root) or not output.is_file():
                raise ValueError("Worker output must exist inside this queue root")
            record.update(
                status="complete" if result["complete"] else "paused",
                chunks=record["chunks"] + 1,
                output_csv=str(output.relative_to(root)).replace("\\", "/"),
                complete=bool(result["complete"]),
            )
            print(source["video_id"], record["status"], "chunks", record["chunks"], flush=True)
        except Exception as error:  # noqa: BLE001 -- Persist worker errors; never silently retry.
            message = str(error)
            blocked = any(
                token in message.casefold()
                for token in ["sign-in", "sign in", "verification", "not a bot"]
            )
            record.update(
                status="blocked" if blocked else "failed",
                error_type=type(error).__name__,
                error=message[:1500],
            )
            print(record["status"], source["video_id"], message, flush=True)
        atomic_json(path, state)
    return state
''',
    "context.py": r'''"""Versioned HUD phase proposals; clocks and display colors are not ground truth."""

import json
import re
from collections import defaultdict
from pathlib import Path

from .pov import read_table, write_table

VERSION = "hud_context_v1"
LIVE_ROUND_SECONDS = 100


def clock_seconds(text):
    match = re.fullmatch(r"(\d{1,2})[:.](\d{2})", text.strip())
    if not match or int(match[2]) >= 60:
        return None
    return int(match[1]) * 60 + int(match[2])


def phase(items, layouts=(), broadcast_state="gameplay"):
    """Never classify a short countdown, or the string 1:30, by value alone."""
    central, texts = [], []
    rounds = set()
    for item in items:
        confidence = float(item.get("confidence") or 0)
        text = item.get("text", "").strip().casefold()
        if confidence >= 0.5:
            texts.append(text)
        width, height = float(item.get("frame_width") or 0), float(item.get("frame_height") or 0)
        if width <= 0 or height <= 0:
            continue
        x = (float(item.get("x1") or 0) + float(item.get("x2") or 0)) / 2 / width
        y = (float(item.get("y1") or 0) + float(item.get("y2") or 0)) / 2 / height
        if not (0.4 <= x <= 0.6 and 0 <= y <= 0.10):
            continue
        seconds = clock_seconds(text)
        if seconds is not None and confidence >= 0.3:
            central.append(seconds)
        round_match = re.fullmatch(r"round\s*(\d{1,2})", text)
        if round_match and confidence >= 0.2 and int(round_match[1]) > 0:
            rounds.add(int(round_match[1]))
    clocks = set(central)
    seconds = next(iter(clocks)) if len(clocks) == 1 else None
    round_number = next(iter(rounds)) if len(rounds) == 1 else ""
    combined = " ".join(texts)
    label, reason = "unknown", "Insufficient phase evidence; no state carried over a highlight cut"
    if re.search(r"\breplay\b", combined) or broadcast_state == "replay":
        label, reason = "replay", "Replay marker or segment decision"
    elif re.search(
        r"\b(technical pause|tech pause|technical time ?out|tech time ?out)\b", combined
    ):
        label, reason = "technical_pause", "Explicit technical-pause marker"
    elif re.search(r"\b(tactical (?:time ?out|pause)|time ?out)\b", combined):
        label, reason = (
            "tactical_timeout",
            "Explicit timeout marker; duration not inferred from the countdown",
        )
    elif re.search(r"\b(match paused|game paused|halftime|half time)\b", combined):
        label, reason = "pause_or_halftime", "Explicit pause/halftime marker"
    elif re.search(r"\b(buy phase|purchase phase)\b", combined):
        label, reason = "buy_phase", "Explicit buy-phase text"
    elif re.search(r"\b(spike planted)\b", combined):
        label, reason = "postplant", "Explicit spike-planted text; plant timestamp unknown"
    elif re.search(r"\b(round won|round lost|flawless|thrifty|ace|team ace)\b", combined):
        label, reason = "round_end_candidate", "Round-result text; verify result overlay"
    elif broadcast_state not in {"gameplay", "unfiltered"}:
        reason = "No admitted match HUD; do not infer phase from off-game clocks"
    elif len(clocks) > 1:
        reason = "Conflicting central clocks"
    elif seconds is not None and 0 < seconds <= 45 and set(layouts) == {"buy"}:
        label, reason = (
            "buy_phase_candidate",
            "Short central countdown and buy-layout roster; barrier/text review still needed",
        )
    elif (
        seconds is not None
        and 45 < seconds <= LIVE_ROUND_SECONDS
        and set(layouts) == {"sidebar"}
        and round_number
    ):
        label, reason = (
            "active_round_candidate",
            "Long round clock, central round label and sidebar HUD; unseen pause remains possible",
        )
    elif seconds is not None and seconds <= LIVE_ROUND_SECONDS:
        reason = "Countdown alone cannot separate buy, live, pause and result phases"
    return {
        "phase": label,
        "reason": reason,
        "clock_seconds": seconds if seconds is not None else "",
        "round_number": round_number,
        "review_status": "proposed",
        "rule_version": VERSION,
    }


def team_side(
    team_id, round_number, team_ids, first_attacking_team=None, overtime_first_attacking_team=None
):
    """Use reviewed starting sides, never HUD colors or screen left/right."""
    if (
        len(team_ids) != 2
        or len(set(team_ids)) != 2
        or team_id not in team_ids
        or not isinstance(round_number, int)
        or isinstance(round_number, bool)
        or round_number < 1
    ):
        return "unknown"
    if round_number <= 24:
        initial = first_attacking_team
        swap = round_number > 12
    else:
        initial = overtime_first_attacking_team
        swap = (round_number - 25) % 2 == 1
    if initial not in team_ids:
        return "unknown"
    attacker = next(team for team in team_ids if team != initial) if swap else initial
    return "attack" if team_id == attacker else "defense"


def export_context(observations_csv, segments_csv, roster_csv):
    observations, layouts = defaultdict(list), defaultdict(set)
    for row in read_table(observations_csv):
        if row["kind"] == "ocr":
            observations[row["timestamp_ms"]].append(row)
    for row in read_table(roster_csv):
        layouts[row["timestamp_ms"]].add(row["layout"])
    rows = []
    for segment in read_table(segments_csv):
        stamp = segment["timestamp_ms"]
        result = phase(observations[stamp], layouts[stamp], segment["segment_type"])
        rows.append(
            dict(
                video_id=segment["video_id"],
                source_sha256=segment["source_sha256"],
                timestamp_ms=stamp,
                frame_index=segment["frame_index"],
                **result,
            )
        )
    fields = [
        "video_id",
        "source_sha256",
        "timestamp_ms",
        "frame_index",
        "phase",
        "reason",
        "clock_seconds",
        "round_number",
        "review_status",
        "rule_version",
    ]
    return write_table(Path(observations_csv).with_name("round_context.csv"), rows, fields)


def export_minimap_review(segments_csv):
    """Save review-only crops for the inspected source layout, not player tracks."""
    from PIL import Image

    path = Path(segments_csv)
    rows = []
    for segment in read_table(path):
        if segment["video_id"] != "omjZXmc7_4A":
            continue  # Other broadcasts require their own reviewed crop profile.
        details = json.loads(segment["text"])
        relative = details.get("review_image")
        if not relative:
            continue
        source = (path.parent / relative).resolve()
        if not source.is_relative_to(path.parent.resolve()) or not source.is_file():
            continue
        target = path.parent / "minimap_review" / f"frame_{segment['frame_index']}.jpg"
        target.parent.mkdir(exist_ok=True)
        with Image.open(source) as image:
            width, height = image.size
            box = (0, round(height * 0.04), round(width * 0.24), round(height * 0.36))
            image.crop(box).save(target, quality=95)
        rows.append(
            {
                "video_id": segment["video_id"],
                "source_sha256": segment["source_sha256"],
                "timestamp_ms": segment["timestamp_ms"],
                "image": str(target.relative_to(path.parent)).replace("\\", "/"),
                "crop_profile": "inspected_vct_1080_minimap_v1",
                "crop_review_status": "proposed",
                "map_id": "",
                "player_id": "",
                "team_id": "",
                "tactical_side": "unknown",
                "marker_x_normalized": "",
                "marker_y_normalized": "",
                "map_region": "",
                "world_x": "",
                "world_y": "",
                "player_visibility": "unknown",
                "reviewer_id": "",
                "notes": "Inspect map presence and crop; no marker detector or identity assignment",
            }
        )
    fields = list(rows[0]) if rows else ["video_id", "timestamp_ms", "image", "crop_review_status"]
    return write_table(path.with_name("minimap_review_suggestions.csv"), rows, fields)
''',
    "entities.py": r'''"""Conservative entity proposals. Co-occurrence is never a player-agent assignment."""

import re
import unicodedata

KINDS = {"player", "team", "map", "character_agent"}


def normalize(value):
    return " ".join(re.findall(r"\w+", unicodedata.normalize("NFKC", value).casefold()))


def validate_catalog(catalog):
    if not isinstance(catalog, list):
        raise TypeError("Catalog must be a list")
    seen = set()
    for entry in catalog:
        if set(entry) != {"kind", "id", "aliases"} or entry["kind"] not in KINDS:
            raise ValueError("Catalog entries require kind, id and aliases")
        if not isinstance(entry["id"], str) or not entry["id"].strip():
            raise ValueError("Entity ID must be a nonempty string")
        if (entry["kind"], entry["id"]) in seen:
            raise ValueError("Duplicate entity ID")
        seen.add((entry["kind"], entry["id"]))
        if not isinstance(entry["aliases"], list) or not entry["aliases"]:
            raise ValueError("Aliases must be a nonempty list")
        if any(not isinstance(alias, str) or not normalize(alias) for alias in entry["aliases"]):
            raise ValueError("Aliases must contain readable text")
    return catalog


def resolve(proposals, catalog):
    """Match complete OCR strings or explicit detector labels; reject ambiguous aliases."""
    index = {}
    for entry in validate_catalog(catalog):
        for alias in entry["aliases"]:
            index.setdefault(normalize(alias), set()).add((entry["kind"], entry["id"]))
    result = []
    for proposal in proposals:
        text = proposal.get("text", "") if proposal["kind"] == "ocr" else proposal["label"]
        matches = index.get(normalize(text), set())
        if len(matches) != 1:
            continue
        kind, identity = next(iter(matches))
        entity = dict(proposal, kind=f"entity.{kind}", label=identity)
        entity[
            {
                "player": "player_id",
                "team": "team_id",
                "map": "map_id",
                "character_agent": "character_agent_id",
            }[kind]
        ] = identity
        result.append(entity)
    return result
''',
    "ocr_retry.py": r'''"""Retry only unresolved roster fields, preserving all original OCR evidence."""

from collections import Counter

from .roster import parse_frame

RETRY_VERSION = "crop_retry_v1"


def retry_regions(rows, width, height, catalog=None):
    roster = parse_frame(rows, width, height, catalog)
    regions = []
    for side, box in [("left", (710, 0, 795, 55)), ("right", (1125, 0, 1200, 55))]:
        if not any(row["side"] == side and row["team_id"] for row in roster):
            regions.append(("team", box))
    layouts = Counter(row["layout"] for row in roster)
    # Do not select a layout when both plausible layouts have recognized names.
    if len(layouts) == 1 and next(iter(layouts.values())) >= 3:
        layout = next(iter(layouts))
        existing = {(row["side"], row["slot"]) for row in roster}
        for side in ["left", "right"]:
            for slot in range(1, 6):
                if (side, slot) in existing:
                    continue
                if layout == "buy":
                    y = 783 + (slot - 1) * 56
                    box = (645, y, 815, y + 53) if side == "left" else (1165, y, 1275, y + 53)
                else:
                    y = 550 + (slot - 1) * 106
                    box = (77, y, 255, y + 45) if side == "left" else (1725, y, 1841, y + 45)
                regions.append(("player_name", box))
    return [
        (
            kind,
            tuple(
                round(value * (width / 1920 if i % 2 == 0 else height / 1080))
                for i, value in enumerate(box)
            ),
        )
        for kind, box in regions
    ]


def retry(frame, reader, rows, catalog=None):
    import cv2

    height, width = frame.shape[:2]
    output = []
    for kind, (x1, y1, x2, y2) in retry_regions(rows, width, height, catalog):
        crop = frame[y1:y2, x1:x2]
        enlarged = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
        for box, text, confidence in reader.readtext(enlarged):
            xs, ys = zip(*box, strict=True)
            output.append(
                {
                    "kind": "ocr",
                    "label": f"retry.{kind}",
                    "text": text,
                    "confidence": float(confidence),
                    "x1": x1 + min(xs) / 3,
                    "y1": y1 + min(ys) / 3,
                    "x2": x1 + max(xs) / 3,
                    "y2": y1 + max(ys) / 3,
                }
            )
    return output
''',
    "pipeline.py": r'''"""YouTube acquisition, sampled OCR/detection proposals, resumable CSV exports."""

import csv
import hashlib
import json
import math
import re
import sqlite3
from functools import lru_cache
from pathlib import Path
from urllib.parse import parse_qs, urlparse

VERSION = "video_v6"
FIELDS = [
    "observation_id",
    "job_id",
    "video_id",
    "source_url",
    "source_sha256",
    "model_sha256",
    "frame_index",
    "timestamp_ms",
    "kind",
    "label",
    "text",
    "confidence",
    "x1",
    "y1",
    "x2",
    "y2",
    "review_status",
    "player_id",
    "team_id",
    "map_id",
    "character_agent_id",
    "pipeline_version",
    "segment_type",
    "processing_action",
    "frame_width",
    "frame_height",
]


def channel_videos(
    url, limit=5, section="videos", cookie_file=None, js_runtime=None, export_root=None
):
    """List a bounded channel page without downloading media."""
    from yt_dlp import YoutubeDL
    from yt_dlp.utils import DownloadError

    from .acquisition import download_error_message, youtube_options

    parsed = urlparse(url)
    if (
        parsed.scheme != "https"
        or parsed.hostname not in {"youtube.com", "www.youtube.com"}
        or not re.fullmatch(r"/@[A-Za-z0-9_.-]+/?", parsed.path)
    ):
        raise ValueError("Expected an HTTPS YouTube @channel URL")
    if not 1 <= limit <= 50 or section not in {"videos", "streams"}:
        raise ValueError("Limit must be 1–50 and section videos or streams")
    target = f"https://www.youtube.com{parsed.path.rstrip('/')}/{section}"
    options = {
        "extract_flat": True,
        "playlistend": limit,
        "skip_download": True,
        **youtube_options(cookie_file, js_runtime, export_root),
    }
    try:
        with YoutubeDL(options) as ydl:
            listing = ydl.extract_info(target, download=False)
    except DownloadError as error:
        raise RuntimeError(download_error_message(error)) from None
    return [
        {
            "video_id": item["id"],
            "title": item.get("title", ""),
            "url": f"https://www.youtube.com/watch?v={item['id']}",
        }
        for item in listing.get("entries", [])
        if item and re.fullmatch(r"[A-Za-z0-9_-]{11}", item.get("id", ""))
    ][:limit]


def video_id(url):
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or host not in {"youtube.com", "www.youtube.com", "youtu.be"}:
        raise ValueError("Use an HTTPS YouTube video link")
    value = (
        parsed.path.strip("/") if host == "youtu.be" else parse_qs(parsed.query).get("v", [""])[0]
    )
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", value):
        raise ValueError("Expected a single watch?v= or youtu.be video link")
    return value


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


@lru_cache(maxsize=4)
def source_digest(path, size, modified_ns):
    """Hash immutable completed media once per size/mtime during a notebook session."""
    digest = sha256(path)
    stat = Path(path).stat()
    if (stat.st_size, stat.st_mtime_ns) != (size, modified_ns):
        raise RuntimeError("Source changed while computing its checksum")
    return digest


def download(url, root, cookie_file=None, js_runtime=None):
    """Download a public video into the notebook runtime; do not bypass access restrictions."""
    from yt_dlp import YoutubeDL
    from yt_dlp.utils import DownloadError

    from .acquisition import cached_video, download_error_message, youtube_options

    key = video_id(url)
    folder = Path(root) / "sources" / key
    folder.mkdir(parents=True, exist_ok=True)
    existing = cached_video(folder, key)
    if existing:
        print("Reusing completed video from cloud cache; no YouTube request needed.")
        return existing
    options = {
        "noplaylist": True,
        "format": "bv*[height<=1080]+ba/b[height<=1080]",
        "merge_output_format": "mp4",
        "outtmpl": str(folder / "video.%(ext)s"),
        "writeinfojson": True,
        "continuedl": True,
        **youtube_options(cookie_file, js_runtime, root),
    }
    try:
        with YoutubeDL(options) as downloader:
            info = downloader.extract_info(f"https://www.youtube.com/watch?v={key}", download=True)
            requested = info.get("requested_downloads") or []
            candidates = [Path(info.get("filepath") or downloader.prepare_filename(info))]
            candidates += [Path(item["filepath"]) for item in requested if item.get("filepath")]
            candidates = [
                folder / "video.mp4",
                *candidates,
                folder / "video.webm",
                folder / "video.mkv",
            ]
            for candidate in candidates:
                if candidate.is_file() and not re.search(r"\.f\d+\.", candidate.name):
                    return candidate
    except DownloadError as error:
        raise RuntimeError(download_error_message(error)) from None
    raise RuntimeError("Download completed without a readable merged video")


def write_csv(path, rows):
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=FIELDS, extrasaction="raise")
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(target)


class Job:
    """One SQLite file per job. Commit each sampled frame, including empty frames."""

    def __init__(self, root, identity):
        self.identity = identity
        self.id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]
        self.folder = Path(root) / "jobs" / self.id
        self.folder.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.folder / "state.sqlite")
        self.db.execute(
            "CREATE TABLE IF NOT EXISTS frames (number INTEGER PRIMARY KEY, rows TEXT NOT NULL)"
        )
        (self.folder / "manifest.json").write_text(json.dumps(identity, indent=2), encoding="utf-8")

    def done(self, number):
        return (
            self.db.execute("SELECT 1 FROM frames WHERE number=?", (number,)).fetchone() is not None
        )

    def restore_gate(self, number, gate):
        encoded = self.db.execute("SELECT rows FROM frames WHERE number=?", (number,)).fetchone()[0]
        for row in json.loads(encoded):
            if row["kind"] == "segment":
                gate.restore(json.loads(row["text"])["gate_state"])

    def records(self, number, timestamp_ms, proposals):
        rows = []
        for i, proposal in enumerate(proposals):
            row = dict.fromkeys(FIELDS, "")
            row.update(proposal)
            row.update(
                observation_id=f"{self.id}:{number}:{i}",
                job_id=self.id,
                video_id=self.identity["video_id"],
                source_url=self.identity["source_url"],
                source_sha256=self.identity["source_sha256"],
                model_sha256=self.identity["model_sha256"],
                frame_index=number,
                timestamp_ms=timestamp_ms,
                review_status="proposed",
                pipeline_version=VERSION,
            )
            rows.append(row)
        return rows

    def commit(self, number, timestamp_ms, proposals):
        rows = self.records(number, timestamp_ms, proposals)
        with self.db:
            self.db.execute("INSERT INTO frames VALUES (?, ?)", (number, json.dumps(rows)))

    def export(self):
        def rows(segment=False):
            for (encoded,) in self.db.execute("SELECT rows FROM frames ORDER BY number"):
                yield from (
                    row for row in json.loads(encoded) if (row["kind"] == "segment") == segment
                )

        path = self.folder / "observations.csv"
        write_csv(path, rows())
        write_csv(self.folder / "segments.csv", rows(segment=True))
        from .segments import export_review_queue

        export_review_queue(self.folder / "segments.csv")
        from .roster import export_rosters

        export_rosters(path, catalog=self.identity.get("roster_catalog"))
        from .context import export_context

        export_context(path, self.folder / "segments.csv", self.folder / "roster_observations.csv")
        if getattr(self, "portrait_matcher", None) is not None:
            from .portraits import classify_rosters

            classify_rosters(path.with_name("roster_observations.csv"), self.portrait_matcher)
        return path

    def close(self):
        self.db.close()


def process(
    url,
    root,
    weights=None,
    interval_seconds=2.0,
    device="cpu",
    end_seconds=None,
    catalog=None,
    segment_filter=True,
    confirm_samples=2,
    save_portrait_crops=True,
    roster_catalog=None,
    crop_ocr_retry=True,
    portrait_reference_csv=None,
    portrait_classifier_weights=None,
    portrait_allowed_classes=None,
    allow_proposed_portrait_references=False,
    review_sample_limit=24,
    cookie_file=None,
    js_runtime=None,
    media_root=None,
    sampling_mode="sequential",
    max_new_samples=None,
    review_every_samples=5,
):
    """OCR bootstrap works without weights; optional YOLO adds custom detector proposals."""
    import cv2
    import easyocr

    from .entities import resolve, validate_catalog
    from .ocr_retry import RETRY_VERSION, retry
    from .roster import PROFILE_VERSION, SOURCE_CATALOG, parse_frame
    from .segments import RULE_VERSION, SegmentGate, classify_probe

    catalog = validate_catalog(catalog or [])
    gate = SegmentGate(confirm_samples)
    if review_every_samples is None:
        raise ValueError("review_every_samples must be a positive integer")
    if sampling_mode not in {"sequential", "seek"}:
        raise ValueError("sampling_mode must be sequential or seek")
    for value, name in [
        (max_new_samples, "max_new_samples"),
        (review_every_samples, "review_every_samples"),
    ]:
        if value is not None and (
            not isinstance(value, int) or isinstance(value, bool) or value < 1
        ):
            raise ValueError(f"{name} must be a positive integer")
    if not isinstance(review_sample_limit, int) or review_sample_limit < 0:
        raise ValueError("review_sample_limit must be a nonnegative integer")
    portrait_matcher = None
    if portrait_reference_csv and portrait_classifier_weights:
        raise ValueError("Choose reference matching or learned classification, not both")
    if portrait_classifier_weights:
        from .portraits import LearnedPortraitClassifier

        if not save_portrait_crops:
            raise ValueError("Portrait classification requires saved portrait crops")
        portrait_matcher = LearnedPortraitClassifier(
            portrait_classifier_weights, device=device, allowed_agent_ids=portrait_allowed_classes
        )
    if portrait_reference_csv:
        from .portraits import PortraitMatcher

        if not save_portrait_crops:
            raise ValueError("Portrait classification requires saved portrait crops")
        portrait_matcher = PortraitMatcher(
            portrait_reference_csv, allow_proposed=allow_proposed_portrait_references
        )

    if not math.isfinite(interval_seconds) or interval_seconds <= 0:
        raise ValueError("Sampling interval must be finite and positive")
    if end_seconds is not None and (not math.isfinite(end_seconds) or end_seconds <= 0):
        raise ValueError("End time must be finite and positive")
    model = None
    model_hash = "none-ocr-bootstrap"
    if weights:
        from ultralytics import YOLO

        model_hash = sha256(weights)
        model = YOLO(str(weights))
        if model.task != "detect":
            raise ValueError("Expected object detection weights")
    source = (
        download(url, media_root or root, cookie_file=cookie_file, js_runtime=js_runtime)
        if cookie_file or js_runtime
        else download(url, media_root or root)
    )
    key = video_id(url)
    roster_catalog = roster_catalog or SOURCE_CATALOG.get(key)
    import importlib.metadata

    versions = {
        name: importlib.metadata.version(name) for name in ["easyocr", "opencv-python-headless"]
    }
    if model:
        versions["ultralytics"] = importlib.metadata.version("ultralytics")
    job = Job(
        root,
        {
            "pipeline_version": VERSION,
            "video_id": key,
            "source_url": f"https://www.youtube.com/watch?v={key}",
            "source_sha256": source_digest(
                str(Path(source).resolve()),
                Path(source).stat().st_size,
                Path(source).stat().st_mtime_ns,
            ),
            "model_sha256": model_hash,
            "interval_seconds": interval_seconds,
            "end_seconds": end_seconds,
            "device": device,
            "packages": versions,
            "source_rights_status": "unconfirmed",
            "is_synthetic": False,
            "catalog": catalog,
            "segment_filter": segment_filter,
            "segment_rule_version": RULE_VERSION,
            "confirm_samples": confirm_samples,
            "roster_profile_version": PROFILE_VERSION,
            "save_portrait_crops": save_portrait_crops,
            "roster_catalog": roster_catalog,
            "crop_ocr_retry": crop_ocr_retry,
            "retry_version": RETRY_VERSION,
            "portrait_matcher": portrait_matcher.identity if portrait_matcher else None,
            "review_sample_limit": review_sample_limit,
            "sampling_mode": sampling_mode,
            "review_every_samples": review_every_samples,
            "sparse_timestamp_method": "frame_index_over_reported_fps",
        },
    )
    job.portrait_matcher = portrait_matcher
    review_folder = job.folder / "segment_review"
    review_count = len(list(review_folder.glob("*.jpg")))
    cap = cv2.VideoCapture(str(source))
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not cap.isOpened() or not math.isfinite(fps) or fps <= 0:
            raise ValueError("Cannot decode video or determine frame rate")
        step = max(1, round(fps * interval_seconds))
        ocr_device = False if device == "cpu" else f"cuda:{device}"
        reader = easyocr.Reader(["en"], gpu=ocr_device)
        number = 0
        processed = 0
        complete = False
        expected_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if sampling_mode == "seek" else 0
        if sampling_mode == "seek" and expected_frames <= 0:
            raise ValueError("Seek sampling requires a readable frame count")
        while end_seconds is None or number / fps < end_seconds:
            if max_new_samples is not None and processed >= max_new_samples:
                break
            if sampling_mode == "seek":
                if number >= expected_frames:
                    complete = True
                    break
                if job.done(number):
                    job.restore_gate(number, gate)
                    number += step
                    continue
                if not cap.set(cv2.CAP_PROP_POS_FRAMES, number):
                    raise RuntimeError("Decoder does not support seek sampling")
            if not cap.grab():
                if sampling_mode == "seek":
                    raise RuntimeError(
                        f"Unexpected decode failure before source end at frame {number}"
                    )
                complete = True
                break
            if number % step == 0 and job.done(number):
                job.restore_gate(number, gate)
            elif number % step == 0:
                ok, frame = cap.retrieve()
                if not ok:
                    raise RuntimeError(f"Frame decode failed at {number}")
                if sampling_mode == "seek":
                    landed = cap.get(cv2.CAP_PROP_POS_FRAMES) - 1
                    if not math.isfinite(landed) or abs(landed - number) > 0.5:
                        raise RuntimeError(
                            "Decoder landed on an unexpected frame; use sequential sampling"
                        )
                proposals = []
                segment_type, action, reason = "unfiltered", "extract", "Filter disabled"
                probe = []
                probe_items = []
                if segment_filter:
                    height = frame.shape[0]
                    probe = reader.readtext(frame[: max(1, round(height * 0.24)), :])
                    for box, text, confidence in probe:
                        xs, ys = zip(*box, strict=True)
                        probe_items.append(
                            {
                                "text": text,
                                "confidence": float(confidence),
                                "x1": float(min(xs)),
                                "y1": float(min(ys)),
                                "x2": float(max(xs)),
                                "y2": float(max(ys)),
                            }
                        )
                    candidate, reason, strong = classify_probe(probe_items, frame.shape[1], height)
                    segment_type, action = gate.update(candidate, strong=strong)
                proposals.append(
                    {
                        "kind": "segment",
                        "label": segment_type,
                        "segment_type": segment_type,
                        "processing_action": action,
                        "text": json.dumps(
                            {
                                "reason": reason,
                                "gate_state": gate.state,
                                "probe_text": [text for _, text, _ in probe],
                                "probe_items": probe_items,
                            }
                        ),
                    }
                )
                readings = reader.readtext(frame) if action == "extract" else []
                for box, text, confidence in readings:
                    xs, ys = zip(*box, strict=True)
                    proposals.append(
                        {
                            "kind": "ocr",
                            "label": "text_region",
                            "text": text,
                            "confidence": float(confidence),
                            "x1": float(min(xs)),
                            "y1": float(min(ys)),
                            "x2": float(max(xs)),
                            "y2": float(max(ys)),
                        }
                    )
                if model and action == "extract":
                    result = model.predict(frame, device=device, verbose=False, conf=0.25)[0]
                    for box in result.boxes:
                        x1, y1, x2, y2 = box.xyxy[0].tolist()
                        proposals.append(
                            {
                                "kind": "detection",
                                "label": result.names[int(box.cls.item())],
                                "confidence": float(box.conf.item()),
                                "x1": x1,
                                "y1": y1,
                                "x2": x2,
                                "y2": y2,
                            }
                        )
                if crop_ocr_retry and action == "extract" and roster_catalog is not None:
                    shape = frame.shape
                    records = job.records(number, round(number / fps * 1000), proposals)
                    proposals.extend(retry(frame, reader, records, catalog=roster_catalog))
                if (
                    number // step % review_every_samples == 0
                    and review_count < review_sample_limit
                ):
                    review_folder.mkdir(exist_ok=True)
                    review_image = review_folder / f"{number:09d}.jpg"
                    if not cv2.imwrite(str(review_image), frame):
                        raise RuntimeError("Could not save segment review frame")
                    review_count += 1
                    details = json.loads(proposals[0]["text"])
                    details["review_image"] = f"segment_review/{review_image.name}"
                    proposals[0]["text"] = json.dumps(details)
                proposals.extend(resolve(proposals, catalog))
                for proposal in proposals:
                    proposal.update(segment_type=segment_type, processing_action=action)
                    shape = getattr(frame, "shape", (1080, 1920))
                    proposal.update(frame_width=shape[1], frame_height=shape[0])
                if save_portrait_crops and action == "extract":
                    # Save crops before committing the frame, so failed writes can retry.
                    records = job.records(number, round(number / fps * 1000), proposals)
                    for roster in parse_frame(
                        records, width=shape[1], height=shape[0], catalog=roster_catalog
                    ):
                        crop = frame[
                            roster["portrait_y1"] : roster["portrait_y2"],
                            roster["portrait_x1"] : roster["portrait_x2"],
                        ]
                        folder = job.folder / "portrait_proposals"
                        folder.mkdir(exist_ok=True)
                        if not cv2.imwrite(
                            str(folder / f"{roster['roster_observation_id']}.jpg"), crop
                        ):
                            raise RuntimeError("Could not save portrait proposal")
                job.commit(number, round(number / fps * 1000), proposals)
                processed += 1
                if processed % 25 == 0:
                    job.export()
                    print(
                        f"Saved {processed} additional sampled frames; source time {number / fps:.1f}s"
                    )
            number += step if sampling_mode == "seek" else 1
        else:
            complete = True
        if number == 0:
            raise RuntimeError("Video decoded no frames")
        progress = job.folder / "progress.json"
        temporary = progress.with_suffix(".json.tmp")
        temporary.write_text(
            json.dumps(
                {
                    "complete": complete,
                    "next_frame_index": number,
                    "new_samples": processed,
                    "sampling_mode": sampling_mode,
                }
            ),
            encoding="utf-8",
        )
        temporary.replace(progress)
        return job.export()
    finally:
        cap.release()
        job.export()
        job.close()


def merge(inputs, output):
    """Deduplicate identical jobs; retain different model versions and reject collisions."""
    rows = {}
    for path in inputs:
        with Path(path).open(newline="", encoding="utf-8") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != FIELDS:
                raise ValueError(f"Incompatible CSV schema: {path}")
            for row in reader:
                key = row["observation_id"]
                if not key:
                    raise ValueError("Missing observation ID")
                if key in rows and rows[key] != row:
                    raise ValueError(f"Conflicting observation: {key}")
                rows[key] = row
    write_csv(output, (rows[key] for key in sorted(rows)))
    return Path(output)
''',
    "portraits.py": r'''"""Small reference-matching baseline with explicit abstention, not calibrated accuracy."""

import csv
import hashlib
import json
import math
from pathlib import Path

REFERENCE_VERSION = "portrait_reference_v1"


def choose(scores, min_score=0.80, min_margin=0.08):
    """Scores are feature cosine similarities, not probabilities."""
    if not 0 <= min_score <= 1 or not 0 <= min_margin <= 2:
        raise ValueError("Invalid reference thresholds")
    best = {}
    for label, score in scores:
        if not label or not math.isfinite(score):
            continue
        best[label] = max(best.get(label, -1), score)
    ranked = sorted(best.items(), key=lambda item: (-item[1], item[0]))
    if len(ranked) < 2:
        return {
            "character_agent_id": "",
            "score": None,
            "margin": None,
            "status": "insufficient_reference_classes",
        }
    (label, score), (_, runner_up) = ranked[:2]
    margin = score - runner_up
    accepted = score >= min_score and margin >= min_margin
    return {
        "character_agent_id": label if accepted else "",
        "score": score,
        "margin": margin,
        "status": "proposed" if accepted else "unknown",
    }


def feature(image):
    import cv2
    import numpy as np

    # Remove lower HUD border, preserve aspect ratio, and normalize contrast.
    gray = cv2.cvtColor(image[: max(1, round(image.shape[0] * 0.85))], cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    ratio = min(32 / width, 32 / height)
    gray = cv2.resize(gray, (max(1, round(width * ratio)), max(1, round(height * ratio))))
    canvas = np.zeros((32, 32), dtype=np.float32)
    y, x = (32 - gray.shape[0]) // 2, (32 - gray.shape[1]) // 2
    canvas[y : y + gray.shape[0], x : x + gray.shape[1]] = gray
    vector = canvas.ravel()
    vector -= vector.mean()
    norm = float(np.linalg.norm(vector))
    return vector / norm if norm > 1e-6 else None


class PortraitMatcher:
    def __init__(self, reference_csv, allow_proposed=False, min_score=0.80, min_margin=0.08):
        import cv2

        self.min_score, self.min_margin = min_score, min_margin
        choose([], min_score, min_margin)
        path = Path(reference_csv).resolve()
        root = path.parent
        self.references = []
        digest = hashlib.sha256(path.read_bytes())
        with path.open(newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                reviewed = (
                    row.get("label_status") == "reviewed"
                    and row.get("independent_review_status") == "reviewed"
                    and bool(row.get("reviewer_id"))
                )
                if not reviewed and not (allow_proposed and row.get("label_status") == "proposed"):
                    continue
                if not row.get("character_agent_id"):
                    continue
                image_path = (root / row["image"]).resolve()
                if not image_path.is_relative_to(root):
                    raise ValueError("Reference image is outside its dataset folder")
                image = cv2.imread(str(image_path))
                if image is None:
                    raise ValueError(f"Unreadable reference image: {image_path.name}")
                vector = feature(image)
                if vector is not None:
                    self.references.append((row["character_agent_id"], vector))
                    digest.update(image_path.read_bytes())
        if len({label for label, _ in self.references}) < 2:
            raise ValueError(
                "Need at least two eligible reference classes; independent review is required by default"
            )
        self.identity = {
            "version": REFERENCE_VERSION,
            "sha256": digest.hexdigest(),
            "allow_proposed": allow_proposed,
            "min_score": min_score,
            "min_margin": min_margin,
            "method": "grayscale_cosine_reference",
        }

    def predict(self, image):
        vector = feature(image)
        scores = (
            []
            if vector is None
            else [(label, float(vector @ ref)) for label, ref in self.references]
        )
        return choose(scores, self.min_score, self.min_margin)


class LearnedPortraitClassifier:
    """Adapter for separately trained YOLO classification weights, with abstention."""

    def __init__(
        self, weights, device="0", min_score=0.80, min_margin=0.15, allowed_agent_ids=None
    ):
        from ultralytics import YOLO

        path = Path(weights)
        self.model = YOLO(str(path))
        if self.model.task != "classify":
            raise ValueError("Portrait weights must be an image classifier, not a HUD detector")
        allowed = set(allowed_agent_ids or ["chamber", "viper", "fade", "omen", "neon", "yoru"])
        allowed |= {"unknown", "other", "occluded"}
        if any(label not in allowed for label in self.model.names.values()):
            raise ValueError(
                "Classifier contains unapproved character classes; do not use generic ImageNet weights for extraction"
            )
        self.device = device
        self.min_score, self.min_margin = min_score, min_margin
        choose([], min_score, min_margin)
        self.identity = {
            "version": "portrait_classifier_v1",
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "method": "yolo_classification",
            "min_score": min_score,
            "min_margin": min_margin,
            "allowed_classes": sorted(allowed),
        }

    def predict(self, image):
        result = self.model.predict(image, device=self.device, verbose=False)[0]
        values = result.probs.data.tolist()
        scores = [(self.model.names[i], float(value)) for i, value in enumerate(values)]
        output = choose(scores, self.min_score, self.min_margin)
        if output["character_agent_id"].casefold() in {"unknown", "other", "occluded"}:
            output.update(character_agent_id="", status="unknown")
        return output


def classify_rosters(roster_csv, matcher):
    import cv2

    path = Path(roster_csv)
    with path.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        fields = list(reader.fieldnames or [])
        rows = list(reader)
    # Keep roster output schema separate from optional portrait enrichment.
    fields = list(rows[0]) if rows else fields
    fields += ["portrait_status", "portrait_score", "portrait_margin", "portrait_model_version"]
    for row in rows:
        crop = path.parent / "portrait_proposals" / (row["roster_observation_id"] + ".jpg")
        image = cv2.imread(str(crop)) if crop.is_file() else None
        prediction = (
            matcher.predict(image)
            if image is not None
            else {"character_agent_id": "", "status": "missing_crop", "score": None, "margin": None}
        )
        row.update(
            character_agent_id=prediction["character_agent_id"],
            portrait_status=prediction["status"],
            portrait_score=prediction["score"],
            portrait_margin=prediction["margin"],
            portrait_model_version=matcher.identity["version"],
        )
    target = path.with_name("roster_with_agents.csv")
    if not fields:
        return None
    with target.with_suffix(".csv.tmp").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    target.with_suffix(".csv.tmp").replace(target)
    (path.parent / "portrait_model.json").write_text(
        json.dumps(matcher.identity, indent=2), encoding="utf-8"
    )
    return target
''',
    "pov.py": r'''"""Offline POV review windows and evidence-gated cue-to-action measurements.

No action detector or psychological inference is implied by these exports.
"""

import csv
import hashlib
import json
import math
import shutil
from pathlib import Path

VERSION = "pov_review_v1"
EVENT_FIELDS = [
    "event_id",
    "window_id",
    "player_id",
    "team_id",
    "map_id",
    "round_id",
    "cue_type",
    "action_type",
    "cue_frame_id",
    "action_frame_id",
    "review_status",
    "reviewer_id",
    "perspective",
    "player_identity_verified",
    "xray_absent",
    "continuous_view",
    "cue_visibility_verified",
    "earlier_cues_unknown",
    "notes",
]


def write_table(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)
    return path


def read_table(path):
    with Path(path).open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def propose_windows(segments_csv, limit=6, seconds=4.0):
    """Bounded coverage samples, not detected events or confirmed player POVs."""
    if not 1 <= limit <= 20 or not math.isfinite(seconds) or not 0 < seconds <= 8:
        raise ValueError("Use 1–20 windows, each at most eight seconds")
    context_file = Path(segments_csv).with_name("round_context.csv")
    excluded = set()
    if context_file.is_file():
        excluded = {
            row["timestamp_ms"]
            for row in read_table(context_file)
            if row["phase"]
            in {
                "buy_phase",
                "buy_phase_candidate",
                "technical_pause",
                "tactical_timeout",
                "pause_or_halftime",
                "round_end_candidate",
                "replay",
            }
        }
    rows = []
    last_end = -1
    for row in read_table(segments_csv):
        if row["segment_type"] != "gameplay" or row["processing_action"] != "extract":
            continue
        if row["timestamp_ms"] in excluded:
            continue
        center = float(row["timestamp_ms"]) / 1000
        start = max(0, center - seconds / 2)
        if start < last_end:
            continue
        rows.append({"start_seconds": start, "end_seconds": start + seconds})
        last_end = start + seconds + 10
        if len(rows) == limit:
            break
    return rows


def extract_windows(source, output_dir, windows, expected_sha256=None):
    """Decode original presentation timestamps with PyAV; no FPS conversion.

    Each completed window is immutable and resumable. A decode/frame-limit failure
    never publishes a partial window. Input size is bounded for a free notebook.
    """
    import av

    from .pipeline import sha256

    if not 1 <= len(windows) <= 20:
        raise ValueError("Select 1–20 bounded windows")
    normalized = []
    for window in windows:
        start, end = float(window["start_seconds"]), float(window["end_seconds"])
        if not all(map(math.isfinite, [start, end])) or start < 0 or not 0 < end - start <= 8:
            raise ValueError("Windows must be finite, nonnegative and at most eight seconds")
        normalized.append((start, end))
    source_hash = sha256(source)
    if expected_sha256 and source_hash != expected_sha256:
        raise ValueError("Source checksum differs from extraction manifest")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifests = []
    for start, end in normalized:
        identity = {
            "source_sha256": source_hash,
            "start_seconds": start,
            "end_seconds": end,
            "schema_version": VERSION,
            "pyav_version": av.__version__,
        }
        key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]
        folder = output_dir / key
        if (folder / "manifest.json").is_file():
            if json.loads((folder / "manifest.json").read_text()) != identity:
                raise ValueError("Window cache identity mismatch")
            cached = read_table(folder / "frames.csv")
            if not cached or any(not (folder / row["image_path"]).is_file() for row in cached):
                raise ValueError("Incomplete window cache; restore or remove this window")
            manifests.append(folder / "frames.csv")
            continue
        temporary = output_dir / (key + ".partial")
        temporary.mkdir(exist_ok=True)
        rows = []
        try:
            with av.open(str(source)) as container:
                stream = container.streams.video[0]
                origin = float(stream.start_time * stream.time_base) if stream.start_time else 0.0
                container.seek(int((start + origin) / stream.time_base), stream=stream)
                previous = None
                for frame in container.decode(stream):
                    if frame.pts is None:
                        raise ValueError("Missing presentation timestamp; cannot measure timing")
                    timestamp = float(frame.pts * frame.time_base) - origin
                    if timestamp < start:
                        continue
                    if timestamp >= end:
                        break
                    if previous is not None and timestamp <= previous:
                        raise ValueError("Non-increasing presentation timestamps")
                    previous = timestamp
                    if len(rows) >= 1000:
                        raise ValueError("Window exceeds 1000-frame safety limit")
                    index = len(rows)
                    name = f"frame_{index:05d}.jpg"
                    frame.to_image().save(temporary / name, quality=90)
                    rows.append(
                        {
                            "frame_id": f"{key}:{index}",
                            "window_id": key,
                            "source_sha256": source_hash,
                            "pts": frame.pts,
                            "time_base": str(frame.time_base),
                            "timestamp_ms": timestamp * 1000,
                            "image_path": name,
                            "schema_version": VERSION,
                        }
                    )
            if len(rows) < 2:
                raise ValueError("Window contains fewer than two decoded frames")
            write_table(temporary / "frames.csv", rows, list(rows[0]))
            (temporary / "manifest.json").write_text(
                json.dumps(identity, indent=2), encoding="utf-8"
            )
            if folder.exists():
                raise ValueError("Incomplete destination window already exists")
            temporary.rename(folder)
        except Exception:
            # Only the verified child created for this bounded window is removed.
            if temporary.parent.resolve() == output_dir.resolve():
                shutil.rmtree(temporary)
            raise
        manifests.append(folder / "frames.csv")
    # Never overwrite a user's annotations on rerun.
    template = output_dir / "pov_events_to_review.csv"
    if not template.exists():
        write_table(
            template,
            [
                dict.fromkeys(EVENT_FIELDS, "")
                | {
                    "window_id": path.parent.name,
                    "review_status": "unreviewed",
                    "earlier_cues_unknown": "true",
                }
                for path in manifests
            ],
            EVENT_FIELDS,
        )
    return manifests


def export_timings(events_csv, frame_manifests, destination):
    """Only reviewed, continuous player POV evidence yields a numeric latency.

    Temporal bounds bracket the first visible cue/action using preceding frames.
    They describe video sampling uncertainty, not physiological reaction time.
    """
    frames, previous = {}, {}
    for manifest in frame_manifests:
        prior = None
        for row in read_table(manifest):
            key = row["frame_id"]
            stamp = float(row["timestamp_ms"])
            if (
                key in frames
                or not math.isfinite(stamp)
                or (prior and stamp <= float(prior["timestamp_ms"]))
            ):
                raise ValueError("Duplicate frame ID or invalid frame timeline")
            frames[key] = row
            previous[key] = prior
            prior = row
    results, seen = [], set()
    for event in read_table(events_csv):
        if not event.get("event_id"):
            continue  # Untouched template row.
        if event["event_id"] in seen:
            raise ValueError("Duplicate event ID")
        seen.add(event["event_id"])
        reason = ""
        cue, action = frames.get(event["cue_frame_id"]), frames.get(event["action_frame_id"])
        checks = [
            "player_identity_verified",
            "xray_absent",
            "continuous_view",
            "cue_visibility_verified",
        ]
        if event["review_status"] != "reviewed" or not event["reviewer_id"]:
            reason = "independent_review_required"
        elif not event["player_id"] or event["perspective"] != "player_pov":
            reason = "unknown_player_or_non_player_pov"
        elif any(event[name] != "true" for name in checks):
            reason = "visibility_or_continuity_not_verified"
        elif event["cue_type"] != "enemy_first_visible" or event["action_type"] not in {
            "shot",
            "ability",
            "disengage",
        }:
            reason = "unsupported_event_type"
        elif not cue or not action:
            reason = "missing_frame_evidence"
        elif (
            any(row["window_id"] != event["window_id"] for row in [cue, action])
            or cue["source_sha256"] != action["source_sha256"]
        ):
            reason = "evidence_source_or_window_mismatch"
        elif not previous[cue["frame_id"]] or not previous[action["frame_id"]]:
            reason = "onset_not_bracketed_by_previous_frame"
        elif float(action["timestamp_ms"]) < float(cue["timestamp_ms"]):
            reason = "action_precedes_cue"
        result = {name: event.get(name, "") for name in EVENT_FIELDS}
        result.update(
            source_sha256=cue["source_sha256"] if cue else "",
            latency_ms="",
            latency_lower_ms="",
            latency_upper_ms="",
            measurement_status="insufficient_evidence",
            exclusion_reason=reason,
            metric="visible_cue_to_action_latency",
            schema_version=VERSION,
        )
        if not reason:
            cue_ms, action_ms = float(cue["timestamp_ms"]), float(action["timestamp_ms"])
            result.update(
                latency_ms=action_ms - cue_ms,
                latency_lower_ms=max(
                    0, float(previous[action["frame_id"]]["timestamp_ms"]) - cue_ms
                ),
                latency_upper_ms=action_ms - float(previous[cue["frame_id"]]["timestamp_ms"]),
                measurement_status="reviewed_video_estimate",
            )
        results.append(result)
    fields = EVENT_FIELDS + [
        "source_sha256",
        "latency_ms",
        "latency_lower_ms",
        "latency_upper_ms",
        "measurement_status",
        "exclusion_reason",
        "metric",
        "schema_version",
    ]
    return write_table(destination, results, fields)
''',
    "roster.py": r'''"""Spatial VCT roster proposals. No membership or pick is inferred from world labels."""

import csv
import hashlib
from collections import defaultdict
from pathlib import Path

from .entities import normalize

PROFILE_VERSION = "vct_1080_v3"
NAMES = [
    "Cryo",
    "bang",
    "Vora",
    "Asuna",
    "Timotino",
    "Chronicle",
    "Derke",
    "Jamppi",
    "Sayonara",
    "PROFEK",
]
# This seed is scoped to the inspected source; it is not a global player directory.
SOURCE_CATALOG = {
    "omjZXmc7_4A": {
        "players": {
            normalize(name): f"source_handle:omjZXmc7_4A:{normalize(name)}" for name in NAMES
        },
        "teams": {"100t": "100t", "i0ot": "100t", "10ot": "100t", "iqot": "100t", "vit": "vit"},
        "maps": {"split": "split", "summit": "summit", "lotus": "lotus"},
    }
}
ROSTER_FIELDS = [
    "roster_observation_id",
    "source_url",
    "video_id",
    "timestamp_ms",
    "frame_index",
    "map_id",
    "team_id",
    "opposing_team_id",
    "player_id",
    "display_name",
    "character_agent_id",
    "side",
    "slot",
    "layout",
    "name_confidence",
    "team_confidence",
    "map_confidence",
    "review_status",
    "evidence_observation_id",
    "portrait_x1",
    "portrait_y1",
    "portrait_x2",
    "portrait_y2",
    "profile_version",
]


def inside(row, region, width, height):
    x = (float(row["x1"]) + float(row["x2"])) / 2 * 1920 / width
    y = (float(row["y1"]) + float(row["y2"])) / 2 * 1080 / height
    x1, y1, x2, y2 = region
    return x1 <= x <= x2 and y1 <= y <= y2


def unique(candidates):
    """Conflicting OCR identities in one region require review, not a best guess."""
    if not candidates or len({value for value, _ in candidates}) != 1:
        return "", ""
    return candidates[0][0], max(confidence for _, confidence in candidates)


def parse_frame(rows, width=1920, height=1080, catalog=None):
    if width <= 0 or height <= 0:
        raise ValueError("Frame dimensions must be positive")
    if not rows:
        return []
    source = rows[0]["video_id"]
    catalog = catalog or SOURCE_CATALOG.get(source)
    if catalog is None:
        return []  # Unknown broadcast needs a reviewed catalog, not guessed identities.
    ocr = [row for row in rows if row["kind"] == "ocr"]
    teams = []
    for region in [(710, 0, 795, 55), (1125, 0, 1200, 55)]:
        candidates = [
            (catalog["teams"][normalize(row["text"])], float(row["confidence"]))
            for row in ocr
            if float(row["confidence"]) >= 0.2
            and normalize(row["text"]) in catalog["teams"]
            and inside(row, region, width, height)
        ]
        teams.append(unique(candidates))
    if teams[0][0] and teams[0][0] == teams[1][0]:
        teams = [("", ""), ("", "")]  # Two scoreboard sides cannot establish one team.
    maps = []
    for row in ocr:
        text = normalize(row["text"])
        if text.startswith("current ") and inside(row, (0, 0, 220, 35), width, height):
            name = text.removeprefix("current ")
            if name in catalog["maps"] and float(row["confidence"]) >= 0.5:
                maps.append((catalog["maps"][name], float(row["confidence"])))
    map_id, map_confidence = unique(maps)
    output = []
    for layout in ["buy", "sidebar"]:
        for side in range(2):
            for slot in range(5):
                if layout == "buy":
                    y = 783 + slot * 56
                    region = (645, y, 815, y + 53) if side == 0 else (1165, y, 1275, y + 53)
                    portrait = (573, y, 641, y + 53) if side == 0 else (1276, y, 1348, y + 53)
                else:
                    y = 550 + slot * 106
                    region = (77, y, 255, y + 45) if side == 0 else (1725, y, 1841, y + 45)
                    portrait = (17, y, 76, y + 47) if side == 0 else (1841, y, 1905, y + 47)
                candidates = [
                    row
                    for row in ocr
                    if normalize(row["text"]) in catalog["players"]
                    and float(row["confidence"]) >= 0.5
                    and inside(row, region, width, height)
                ]
                player, confidence = unique(
                    [
                        (catalog["players"][normalize(row["text"])], float(row["confidence"]))
                        for row in candidates
                    ]
                )
                if not player:
                    continue
                evidence = max(candidates, key=lambda row: float(row["confidence"]))
                identity = f"{evidence['observation_id']}:{PROFILE_VERSION}:{layout}:{side}:{slot}"
                row = {field: "" for field in ROSTER_FIELDS}
                row.update(
                    roster_observation_id=hashlib.sha256(identity.encode()).hexdigest()[:24],
                    source_url=evidence["source_url"],
                    video_id=source,
                    timestamp_ms=evidence["timestamp_ms"],
                    frame_index=evidence["frame_index"],
                    map_id=map_id,
                    map_confidence=map_confidence,
                    team_id=teams[side][0],
                    team_confidence=teams[side][1],
                    opposing_team_id=teams[1 - side][0],
                    player_id=player,
                    display_name=evidence["text"],
                    side=["left", "right"][side],
                    slot=slot + 1,
                    layout=layout,
                    name_confidence=confidence,
                    review_status="proposed",
                    evidence_observation_id=evidence["observation_id"],
                    profile_version=PROFILE_VERSION,
                )
                for field, value in zip(
                    ["portrait_x1", "portrait_y1", "portrait_x2", "portrait_y2"],
                    portrait,
                    strict=True,
                ):
                    row[field] = round(
                        value * (width / 1920 if field.endswith(("x1", "x2")) else height / 1080)
                    )
                output.append(row)
    # Two possible layouts or slots for one identity is ambiguous.
    counts = defaultdict(int)
    for row in output:
        counts[row["player_id"]] += 1
    return [row for row in output if counts[row["player_id"]] == 1]


def export_rosters(observations, output=None, width=1920, height=1080, catalog=None):
    groups = defaultdict(list)
    with Path(observations).open(newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            if row.get("processing_action") == "extract":
                groups[(row["video_id"], int(row["frame_index"]))].append(row)
    result = []
    for key in sorted(groups):
        group = groups[key]
        result.extend(
            parse_frame(
                group,
                width=int(group[0].get("frame_width") or width),
                height=int(group[0].get("frame_height") or height),
                catalog=catalog,
            )
        )
    target = Path(output) if output else Path(observations).with_name("roster_observations.csv")
    temp = target.with_suffix(".csv.tmp")
    with temp.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=ROSTER_FIELDS)
        writer.writeheader()
        writer.writerows(result)
    temp.replace(target)
    return target
''',
    "segments.py": r'''"""Conservative broadcast rules; unknown layouts remain reviewable, never guessed."""

import csv
import json
import re
from pathlib import Path

RULE_VERSION = "hud_rules_v3"


def export_review_queue(segment_csv):
    path = Path(segment_csv)
    fields = [
        "video_id",
        "timestamp_ms",
        "image",
        "suggested_segment",
        "processing_action",
        "reason",
        "reference_segment",
        "review_status",
        "reviewer_id",
    ]
    # Reviewers copy this generated file to segment_review_labels.csv before editing.
    target = path.with_name("segment_review_suggestions.csv")
    temporary = target.with_suffix(".csv.tmp")
    with (
        path.open(newline="", encoding="utf-8") as source,
        temporary.open("w", newline="", encoding="utf-8") as dest,
    ):
        writer = csv.DictWriter(dest, fieldnames=fields)
        writer.writeheader()
        for row in csv.DictReader(source):
            details = json.loads(row["text"])
            if details.get("review_image"):
                writer.writerow(
                    {
                        "video_id": row["video_id"],
                        "timestamp_ms": row["timestamp_ms"],
                        "image": details["review_image"],
                        "suggested_segment": row["segment_type"],
                        "processing_action": row["processing_action"],
                        "reason": details["reason"],
                        "reference_segment": "",
                        "review_status": "pending",
                        "reviewer_id": "",
                    }
                )
    temporary.replace(target)
    return target


def classify(texts):
    """Use a small upper-HUD OCR probe, not a full-frame detector, on every sample."""
    tokens = [text.strip().casefold() for text in texts]
    combined = " ".join(tokens)
    if re.search(r"\breplay\b", combined):
        return "replay", "Replay marker visible"
    if re.search(r"\b(starting lineup|agent select|map veto|map pick|team roster)\b", combined):
        return "roster", "Roster or selection marker visible"
    if re.search(
        r"\b(be right back|we.?ll be right back|starting soon|stream starting|"
        r"technical pause|halftime|half time|match paused|commercial break|"
        r"analyst desk|up next|stay tuned|intermission)\b",
        combined,
    ):
        return "filler", "Break, desk or pause marker visible"
    timer = any(re.fullmatch(r"\d{1,2}[:.]\d{2}", text) for text in tokens)
    # VCT's map-order strip and round label remain visible when score OCR fails.
    round_label = bool(re.search(r"\bround\s*\d{1,2}\b", combined))
    map_order = bool(re.search(r"\bcurrent\s*[:.]\s*\w+", combined)) and bool(
        re.search(r"\bnext\s*[:.]\s*\w+", combined)
    )
    if round_label and map_order:
        return "gameplay", "VCT map-order strip and round label visible"
    scores = sum(bool(re.fullmatch(r"\d{1,2}", text)) for text in tokens)
    if timer and scores >= 2:
        return "gameplay", "Upper HUD timer and two score candidates visible"
    return "uncertain", "No reliable broadcast-state signal"


def classify_probe(items, width, height):
    """Strong same-frame HUD groups can confirm a cut without carrying old state."""
    candidate, reason = classify([item["text"] for item in items if item["confidence"] >= 0.3])
    if candidate in {"filler", "replay", "roster"}:
        return candidate, reason, False

    def located(item, area):
        if not all(key in item for key in ["x1", "y1", "x2", "y2"]):
            return True  # Legacy upper-HUD transcripts: usable only for diagnostic replay.
        x = (item["x1"] + item["x2"]) / 2 / width
        y = (item["y1"] + item["y2"]) / 2 / height
        x1, y1, x2, y2 = area
        return x1 <= x <= x2 and y1 <= y <= y2

    map_text = [
        item["text"].strip().casefold()
        for item in items
        if item["confidence"] >= 0.5 and located(item, (0, 0, 0.34, 0.06))
    ]
    map_order = any(re.match(r"current\s*[:.]\s*\w+", text) for text in map_text) and any(
        re.match(r"next\s*[:.]\s*\w+", text) for text in map_text
    )
    clock = False
    round_label = False
    for item in items:
        text = item["text"].strip().casefold()
        if not located(item, (0.40, 0, 0.60, 0.10)):
            continue
        match = re.fullmatch(r"\d{1,2}[:.](\d{2})", text)
        clock |= bool(match and int(match[1]) < 60 and item["confidence"] >= 0.5)
        round_label |= bool(re.fullmatch(r"round\s*\d{1,2}", text) and item["confidence"] >= 0.2)
    if map_order and (clock or round_label):
        return "gameplay", "Same-frame map strip and central clock/round HUD", True
    return candidate, reason, False


class SegmentGate:
    """Confirm transitions across samples; suppress extraction during ambiguity."""

    def __init__(self, confirm_samples=2):
        if (
            not isinstance(confirm_samples, int)
            or isinstance(confirm_samples, bool)
            or confirm_samples < 1
        ):
            raise ValueError("confirm_samples must be a positive integer")
        self.required = confirm_samples
        self.state = {"stable": "uncertain", "candidate": "uncertain", "count": 0}

    def update(self, candidate, strong=False):
        if candidate not in {"gameplay", "roster", "replay", "filler", "uncertain"}:
            raise ValueError("Unknown segment type")
        count = self.state["count"] + 1 if candidate == self.state["candidate"] else 1
        stable = candidate if count >= self.required else self.state["stable"]
        if strong and candidate == "gameplay":
            stable = candidate
            count = max(count, self.required)
        self.state = {"stable": stable, "candidate": candidate, "count": count}
        # A previously stable match must not admit the first frame of an ad/replay.
        if candidate in {"replay", "filler"}:
            return candidate, "skip"
        if candidate in {"gameplay", "roster"} and stable == candidate and count >= self.required:
            return candidate, "extract"
        return "uncertain", "review"

    def restore(self, state):
        self.state = dict(state)
''',
    "visual_features.py": r'''"""Optional frozen ViT features for reviewed gameplay crops, never facial profiling."""

import hashlib
import json
import math
import re
from pathlib import Path

from .pov import read_table, write_table

VERSION = "visual_features_v1"


class DinoEncoder:
    """Small frozen DINOv2 on a single CUDA GPU; no automatic label generation."""

    def __init__(self, revision, device="cuda:0"):
        import torch
        from transformers import AutoImageProcessor, AutoModel

        if not re.fullmatch(r"[a-f0-9]{40}", revision):
            raise ValueError("Pin an immutable Hugging Face model commit SHA")
        if device != "cuda:0" or not torch.cuda.is_available():
            raise ValueError("This optional encoder requires the single CUDA GPU at cuda:0")
        self.torch = torch
        self.device = device
        self.processor = AutoImageProcessor.from_pretrained(
            "facebook/dinov2-small", revision=revision
        )
        self.model = (
            AutoModel.from_pretrained(
                "facebook/dinov2-small", revision=revision, use_safetensors=True
            )
            .eval()
            .to(device)
        )
        self.identity = {
            "model_id": "facebook/dinov2-small",
            "revision": revision,
            "device": device,
            "precision": "float16_autocast",
        }

    def __call__(self, images):
        torch = self.torch
        inputs = self.processor(images=images, return_tensors="pt").to(self.device)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            features = self.model(**inputs).last_hidden_state[:, 0].float()
            features = torch.nn.functional.normalize(features, dim=-1)
        return features.cpu().tolist()


def export_features(samples_csv, destination, encoder, batch_size=4):
    """Explicit reviewed crop bounds prevent embedding whole broadcasts/facecams.

    Coordinates are pixels in the saved image. Embeddings remain unlabeled.
    Caller supplies an encoder so tests require neither CUDA nor model downloads.
    """
    from PIL import Image

    if not 1 <= batch_size <= 16:
        raise ValueError("Use batch size 1–16 on a single GPU")
    samples = read_table(samples_csv)
    if not 1 <= len(samples) <= 2000:
        raise ValueError("Select 1–2000 reviewed crops")
    ids = set()
    for row in samples:
        if not row["sample_id"] or row["sample_id"] in ids:
            raise ValueError("Sample IDs must be unique and nonempty")
        ids.add(row["sample_id"])
        if row["crop_review_status"] != "reviewed" or row["crop_kind"] not in {
            "gameplay",
            "character_portrait",
        }:
            raise ValueError("Only reviewed gameplay or character portrait crops are allowed")
        if not row["reviewer_id"] or not row["source_sha256"] or not row["timestamp_ms"]:
            raise ValueError("Crop reviewer and source/time provenance are required")
    rows, dimensions = [], None
    for offset in range(0, len(samples), batch_size):
        batch = samples[offset : offset + batch_size]
        images, hashes = [], []
        for row in batch:
            path = Path(row["image_path"])
            if not path.is_absolute():
                path = Path(samples_csv).parent / path
            with Image.open(path) as source:
                box = tuple(int(row[key]) for key in ["x1", "y1", "x2", "y2"])
                if (
                    not 0 <= box[0] < box[2] <= source.width
                    or not 0 <= box[1] < box[3] <= source.height
                ):
                    raise ValueError("Crop coordinates fall outside the saved image")
                crop = source.convert("RGB").crop(box)
                images.append(crop)
                hashes.append(hashlib.sha256(crop.tobytes()).hexdigest())
        vectors = encoder(images)
        if len(vectors) != len(batch):
            raise ValueError("Encoder returned an incorrect batch size")
        for row, vector, crop_hash in zip(batch, vectors, hashes, strict=True):
            vector = [float(value) for value in vector]
            norm = math.sqrt(sum(value * value for value in vector))
            if not vector or not math.isfinite(norm) or norm == 0:
                raise ValueError("Encoder returned an invalid vector")
            dimensions = dimensions or len(vector)
            if len(vector) != dimensions:
                raise ValueError("Embedding dimensions changed")
            rows.append(
                dict(
                    row,
                    embedding=[value / norm for value in vector],
                    crop_sha256=crop_hash,
                    model=encoder.identity,
                    label_status="unlabeled",
                    schema_version=VERSION,
                )
            )
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")
    temporary.replace(destination)
    # Farthest-first review selection is diversity, not uncertainty or confidence.
    chosen, remaining = [], list(range(len(rows)))
    distances = [2.0] * len(rows)
    while remaining and len(chosen) < 100:
        index = max(remaining, key=lambda candidate: distances[candidate])
        chosen.append(index)
        remaining.remove(index)
        for other in remaining:
            distance = 1 - sum(
                a * b
                for a, b in zip(rows[index]["embedding"], rows[other]["embedding"], strict=True)
            )
            distances[other] = min(distances[other], distance)
    queue = [
        {
            "sample_id": rows[index]["sample_id"],
            "diversity_rank": rank + 1,
            "label": "",
            "review_status": "unreviewed",
        }
        for rank, index in enumerate(chosen)
    ]
    write_table(destination.with_suffix(".review_queue.csv"), queue, list(queue[0]))
    return destination
''',
}
package = Path("/tmp/valomind_batch_worker/valomind_video")
package.mkdir(parents=True, exist_ok=True)
for name, source in MODULES.items():
    (package / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(package.parent))
for name in list(sys.modules):
    if name == "valomind_video" or name.startswith("valomind_video."):
        del sys.modules[name]
from valomind_video.pipeline import process, download, VERSION
from valomind_video.batch import run_queue, atomic_json, validate_sources, OFFICIAL_CHANNEL
from valomind_video.acquisition import (
    ensure_runtime,
    cached_video,
    restore_source_archive,
    youtube_options,
)
from valomind_video.pov import read_table, write_table

JS_RUNTIME = ensure_runtime()
WORKER_HASH = hashlib.sha256(json.dumps(MODULES, sort_keys=True).encode()).hexdigest()
import torch
import importlib.metadata

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU in runtime settings before continuing.")
torch.cuda.set_device(0)
DEVICE = "0"
print("Using only CUDA GPU 0:", torch.cuda.get_device_name(0))
probe = torch.ones(1, device="cuda:0") * 2
torch.cuda.synchronize(0)
assert probe.item() == 2
del probe

In [ ]:
# Keep these values unchanged to resume. New parameters require a new RUN_NAME.
RUN_NAME = "seoul_playoffs_batch_v1"
IS_KAGGLE = Path("/kaggle").exists()
if IS_KAGGLE:
    # Kaggle: Save Version with outputs; /kaggle/working is NOT durable across fresh sessions.
    PERSISTENT_ROOT = Path("/kaggle/working/ValoMind")
    MEDIA_ROOT = Path("/kaggle/temp/valomind_media")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    PERSISTENT_ROOT = Path("/content/drive/MyDrive/ValoMind")
    MEDIA_ROOT = Path("/content/valomind_media")
WORK_ROOT = PERSISTENT_ROOT / RUN_NAME
WORK_ROOT.mkdir(parents=True, exist_ok=True)
MEDIA_ROOT.mkdir(parents=True, exist_ok=True)

INTERVAL_SECONDS = 10.0  # Sparse scan of the ENTIRE broadcast, not a two-minute preview.
CHUNK_SAMPLES = 100  # Checkpoint after at most 100 additional probes per worker call.
RUN_BUDGET_MINUTES = 90  # Checked between chunks; a download/chunk can exceed this budget.
REVIEW_IMAGES_PER_VIDEO = 240  # Spread over the broadcast, includes filler/unknown samples.
PURGE_TEMP_VIDEO_AFTER_COMPLETION = True  # Only completed, managed temporary video files.
DENSE_WINDOWS_PER_VIDEO = 0  # Optional 1–2 two-second native-frame windows; adds image storage.
COOKIE_FILE = None  # Own authorized Netscape cookies, outside WORK_ROOT and MEDIA_ROOT.
RETRY_ERRORS = (
    False  # Set True only AFTER fixing sign-in/storage/runtime errors, then return False.
)
SOURCE_ARCHIVES = {}  # Optional {video_id: '/content/drive/MyDrive/<source_archive>.zip'}.
# The old highlight ZIP cannot restore any of these new videos.
ROSTER_CATALOGS = {}  # Optional per-video reviewed aliases/layout-compatible catalog. No global 100T/VIT aliases.
if COOKIE_FILE:
    cookie_path = Path(COOKIE_FILE).resolve()
    if cookie_path.is_relative_to(WORK_ROOT.resolve()) or cookie_path.is_relative_to(
        MEDIA_ROOT.resolve()
    ):
        raise ValueError("Keep cookies outside both export and temporary-media roots.")
if DENSE_WINDOWS_PER_VIDEO not in {0, 1, 2}:
    raise ValueError("Dense windows must be 0, 1, or 2.")
if DENSE_WINDOWS_PER_VIDEO:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av>=14,<17"], check=True)
SOURCES = [
    {
        "video_id": "7t-guum-eZ4",
        "url": "https://www.youtube.com/watch?v=7t-guum-eZ4",
        "title": "EDG vs. TE - VALORANT Champions Seoul - Playoffs",
        "duration_seconds": 18345,
        "channel_id": "UCA1d3HFGFUmkKr2JIUA5Vlw",
        "channel_name": "VALORANT Champions Tour",
        "is_live_content": True,
        "is_live_now": False,
        "publish_date": "2024-08-14T05:58:35-07:00",
        "metadata_checked_date": "2026-10-21",
        "dataset_role": "train_proposed",
    },
    {
        "video_id": "Z_NvZ7kVffc",
        "url": "https://www.youtube.com/watch?v=Z_NvZ7kVffc",
        "title": "TH vs. FNC - VALORANT Champions Seoul - Playoffs Day 2",
        "duration_seconds": 20119,
        "channel_id": "UCA1d3HFGFUmkKr2JIUA5Vlw",
        "channel_name": "VALORANT Champions Tour",
        "is_live_content": True,
        "is_live_now": False,
        "publish_date": "2024-08-15T06:16:11-07:00",
        "metadata_checked_date": "2026-10-21",
        "dataset_role": "train_proposed",
    },
    {
        "video_id": "zPZP_ja5iIU",
        "url": "https://www.youtube.com/watch?v=zPZP_ja5iIU",
        "title": "FNC vs. G2 - VALORANT Champions Seoul - Playoffs Day 3",
        "duration_seconds": 17060,
        "channel_id": "UCA1d3HFGFUmkKr2JIUA5Vlw",
        "channel_name": "VALORANT Champions Tour",
        "is_live_content": True,
        "is_live_now": False,
        "publish_date": "2024-08-16T05:28:11-07:00",
        "metadata_checked_date": "2026-10-21",
        "dataset_role": "train_proposed",
    },
    {
        "video_id": "68LiZLjgwrI",
        "url": "https://www.youtube.com/watch?v=68LiZLjgwrI",
        "title": "TH vs. LEV - VALORANT Champions Seoul - Playoffs Day 4",
        "duration_seconds": 23810,
        "channel_id": "UCA1d3HFGFUmkKr2JIUA5Vlw",
        "channel_name": "VALORANT Champions Tour",
        "is_live_content": True,
        "is_live_now": False,
        "publish_date": "2024-08-17T07:34:13-07:00",
        "metadata_checked_date": "2026-10-21",
        "dataset_role": "validation_reserved",
    },
    {
        "video_id": "tj3tfTcrWTI",
        "url": "https://www.youtube.com/watch?v=tj3tfTcrWTI",
        "title": "LEV vs. TH - VALORANT Champions Seoul - Lower Bracket Final",
        "duration_seconds": 19051,
        "channel_id": "UCA1d3HFGFUmkKr2JIUA5Vlw",
        "channel_name": "VALORANT Champions Tour",
        "is_live_content": True,
        "is_live_now": False,
        "publish_date": "2024-08-24T05:19:04-07:00",
        "metadata_checked_date": "2026-10-21",
        "dataset_role": "test_reserved",
    },
]
# To assign this notebook a subset, use e.g. SOURCES = SOURCES[:1] and a different RUN_NAME.
validate_sources(SOURCES)
atomic_json(WORK_ROOT / "source_catalog.json", {"sources": SOURCES})
(WORK_ROOT / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8"
)
for source in SOURCES:
    seconds = source["duration_seconds"]
    print(
        source["video_id"],
        f"{seconds // 3600}:{seconds // 60 % 60:02d}:{seconds % 60:02d}",
        source["dataset_role"],
        source["title"],
    )
print("Persistent output:", WORK_ROOT)
print("Temporary videos:", MEDIA_ROOT)

## Acquisition and continuation
The next cell downloads one archive at a time, processes bounded chunks, and keeps the
same source active until it finishes or the run budget ends. Completed jobs are skipped.
If YouTube requests verification, the queue stops and records the error instead of retrying
other accounts, proxies or all five URLs. Configure your own authorized cookies or restore
the matching source archive from Drive; then explicitly enable RETRY_ERRORS once.

YouTube metadata/channel IDs are checked again at runtime. The 2024 roster/HUD has not
been calibrated against our earlier video. Unknown identities remain unknown; raw OCR,
probe decisions and uniformly sampled review images are the initial collection targets.


In [ ]:
def validate_metadata(info, source):
    if info.get("id") != source["video_id"] or info.get("channel_id") != OFFICIAL_CHANNEL:
        raise ValueError("Video metadata does not match the selected official source.")
    if info.get("is_live") or info.get("live_status") == "is_live":
        raise ValueError("Wait for the livestream to finish; archived videos only.")
    duration = float(info.get("duration") or 0)
    if not duration or abs(duration - source["duration_seconds"]) > max(5, duration * 0.01):
        raise ValueError("Source duration changed materially; inspect before processing.")


def collect_review_samples(output, source):
    from PIL import Image

    rows = []
    for segment in read_table(output.parent / "segments.csv"):
        details = json.loads(segment["text"])
        relative = details.get("review_image")
        if not relative:
            continue
        image_path = (output.parent / relative).resolve()
        if not image_path.is_relative_to(output.parent.resolve()) or not image_path.is_file():
            continue
        crop = output.parent / "minimap_candidates" / image_path.name
        crop.parent.mkdir(exist_ok=True)
        with Image.open(image_path) as image:
            width, height = image.size
            image.crop((0, round(height * 0.04), round(width * 0.24), round(height * 0.36))).save(
                crop, quality=95
            )
        rows.append(
            {
                "sample_id": segment["observation_id"],
                "video_id": source["video_id"],
                "source_sha256": segment["source_sha256"],
                "timestamp_ms": segment["timestamp_ms"],
                "image": relative,
                "segment_suggestion": segment["segment_type"],
                "dataset_role": source["dataset_role"],
                "source_group": source["video_id"],
                "label": "",
                "review_status": "unreviewed",
                "reviewer_id": "",
                "training_eligible": "false",
                "player_id": "",
                "team_id": "",
                "character_agent_id": "",
                "map_id": "",
                "minimap_candidate": str(crop.relative_to(output.parent)).replace("\\", "/"),
                "crop_review_status": "unreviewed_uncalibrated_region",
            }
        )
    fields = list(rows[0]) if rows else ["sample_id", "video_id", "image", "review_status"]
    write_table(output.parent / "unlabeled_samples.csv", rows, fields)
    atomic_json(
        output.parent / "collection_summary.json",
        {
            "video_id": source["video_id"],
            "review_images": len(rows),
            "reviewed_player_catalog_supplied": bool(ROSTER_CATALOGS.get(source["video_id"])),
            "identity_status": "proposals_only"
            if ROSTER_CATALOGS.get(source["video_id"])
            else "unknown_no_reviewed_catalog",
            "dataset_role": source["dataset_role"],
            "training_performed": False,
        },
    )


def worker(source):
    key = source["video_id"]
    if key in SOURCE_ARCHIVES:
        restore_source_archive(SOURCE_ARCHIVES[key], MEDIA_ROOT, key)
    folder = MEDIA_ROOT / "sources" / key
    video = cached_video(folder, key)
    if video:
        info = json.loads((folder / "video.info.json").read_text(encoding="utf-8"))
        validate_metadata(info, source)
    else:
        if shutil.disk_usage(MEDIA_ROOT).free < 30 * 2**30:
            raise RuntimeError(
                "Less than 30 GiB free temporary disk. Free space before downloading."
            )
        from yt_dlp import YoutubeDL
        from yt_dlp.utils import DownloadError
        from valomind_video.acquisition import download_error_message

        try:
            options = {
                "skip_download": True,
                "noplaylist": True,
                **youtube_options(COOKIE_FILE, JS_RUNTIME, MEDIA_ROOT),
            }
            with YoutubeDL(options) as ydl:
                info = ydl.extract_info(source["url"], download=False)
        except DownloadError as error:
            raise RuntimeError(download_error_message(error)) from None
        validate_metadata(info, source)
        video = download(source["url"], MEDIA_ROOT, cookie_file=COOKIE_FILE, js_runtime=JS_RUNTIME)
    stride = max(
        1, math.ceil(source["duration_seconds"] / INTERVAL_SECONDS / REVIEW_IMAGES_PER_VIDEO)
    )
    output = process(
        source["url"],
        WORK_ROOT,
        media_root=MEDIA_ROOT,
        device=DEVICE,
        cookie_file=COOKIE_FILE,
        js_runtime=JS_RUNTIME,
        interval_seconds=INTERVAL_SECONDS,
        end_seconds=None,
        sampling_mode="seek",
        max_new_samples=CHUNK_SAMPLES,
        review_sample_limit=REVIEW_IMAGES_PER_VIDEO,
        review_every_samples=stride,
        segment_filter=True,
        save_portrait_crops=False,
        crop_ocr_retry=False,
        roster_catalog=ROSTER_CATALOGS.get(key),
    )
    # Keep source metadata after the large temporary video is purged.
    shutil.copyfile(folder / "video.info.json", output.parent / "source_metadata.json")
    collect_review_samples(output, source)
    progress = json.loads((output.parent / "progress.json").read_text())
    if progress["complete"] and DENSE_WINDOWS_PER_VIDEO:
        from valomind_video.pov import extract_windows

        candidates = [
            row
            for row in read_table(output.parent / "segments.csv")
            if row["segment_type"] == "gameplay" and row["processing_action"] == "extract"
        ]
        if candidates:
            indices = sorted(
                {
                    min(len(candidates) - 1, int(len(candidates) * fraction))
                    for fraction in ([0.5] if DENSE_WINDOWS_PER_VIDEO == 1 else [0.25, 0.75])
                }
            )
            windows = [
                {
                    "start_seconds": max(0, float(candidates[index]["timestamp_ms"]) / 1000 - 1),
                    "end_seconds": max(0, float(candidates[index]["timestamp_ms"]) / 1000 - 1) + 2,
                }
                for index in indices
            ]
            extract_windows(
                video,
                output.parent / "pov_review",
                windows,
                expected_sha256=json.loads((output.parent / "manifest.json").read_text())[
                    "source_sha256"
                ],
            )
    if progress["complete"] and PURGE_TEMP_VIDEO_AFTER_COMPLETION:
        # Delete only managed ephemeral video files, never Drive outputs or user archives.
        allowed = Path(
            "/kaggle/temp/valomind_media" if IS_KAGGLE else "/content/valomind_media"
        ).resolve()
        if MEDIA_ROOT.resolve() != allowed or not video.resolve().is_relative_to(allowed):
            raise ValueError("Refusing media cleanup outside the managed temporary root.")
        for suffix in [".mp4", ".webm", ".mkv"]:
            target = folder / ("video" + suffix)
            if target.is_file() and target.resolve().parent == folder.resolve():
                target.unlink()
    return {"output_csv": output, "complete": progress["complete"]}


CONFIGURATION = {
    "pipeline_version": VERSION,
    "worker_hash": WORKER_HASH,
    "ocr_packages": {
        name: importlib.metadata.version(name) for name in ["easyocr", "opencv-python-headless"]
    },
    "interval_seconds": INTERVAL_SECONDS,
    "review_limit": REVIEW_IMAGES_PER_VIDEO,
    "sampling_mode": "seek",
    "roster_catalogs": ROSTER_CATALOGS,
    "dense_windows": DENSE_WINDOWS_PER_VIDEO,
}
state = run_queue(
    SOURCES,
    WORK_ROOT,
    worker,
    CONFIGURATION,
    budget_seconds=RUN_BUDGET_MINUTES * 60,
    retry_errors=RETRY_ERRORS,
)
print(json.dumps(state, indent=2))
print("Rerun THIS cell to continue the same queue; do not change RUN_NAME or configuration.")

## Export partial or completed data
Run this cell whenever you want a snapshot, including after a queue error. It excludes
full source videos and cookies, retains per-job SQLite state and proposed labels, and
does not randomly split neighboring frames. Three broadcasts are train candidates,
one is reserved for validation, one for testing; no supervised training is performed.
Broadcast/source groups remain intact; confirm match identities before final evaluation.


In [ ]:
# Collect only outputs recorded by this queue, not unrelated jobs in other runs.
from valomind_video.pipeline import merge

state = json.loads((WORK_ROOT / "queue_status.json").read_text())
outputs = [
    WORK_ROOT / item["output_csv"]
    for item in state["jobs"].values()
    if item.get("output_csv") and (WORK_ROOT / item["output_csv"]).is_file()
]
if outputs:
    merge(outputs, WORK_ROOT / "merged_observations.csv")
    for filename in ["round_context.csv", "roster_observations.csv", "unlabeled_samples.csv"]:
        paths = [path.parent / filename for path in outputs if (path.parent / filename).is_file()]
        all_rows, fields = [], []
        for path in paths:
            with path.open(newline="", encoding="utf-8") as stream:
                reader = csv.DictReader(stream)
                fields = list(dict.fromkeys(fields + (reader.fieldnames or [])))
                all_rows.extend(reader)
        if fields:
            write_table(
                WORK_ROOT / ("combined_" + filename),
                [{field: row.get(field, "") for field in fields} for row in all_rows],
                fields,
            )
else:
    print("No completed worker chunk yet; archive still contains queue status for diagnosis.")
from zipfile import ZipFile, ZIP_DEFLATED

archive_dir = PERSISTENT_ROOT / "exports"
archive_dir.mkdir(exist_ok=True)
archive = archive_dir / (RUN_NAME + "_results.zip")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in WORK_ROOT.rglob("*"):
        if not path.is_file() or path.suffix.lower() in {".mp4", ".webm", ".mkv"}:
            continue
        if "cookie" in path.name.lower() or path.name == ".env":
            continue
        bundle.write(path, path.relative_to(WORK_ROOT))
print("Snapshot:", archive, "MiB:", round(archive.stat().st_size / 2**20, 1))
# Drive already retains the archive. Set True if you also want a browser download.
DOWNLOAD_SNAPSHOT = False
if DOWNLOAD_SNAPSHOT and not IS_KAGGLE:
    from google.colab import files

    files.download(str(archive))